# HW0: Intro to MUJOCO

**Guidelines:**

1. We will be using Jupyter Notebook, please install necessary libraries. And also install `mujoco` (we suggest using version 3.5.0).

2. You may use LLMs as an assisting tool, but if you generate a large amount of content with it, please also attach your prompts to demonstrate how you used it to arrive at the final answer.

3. You must submit two files to the Gradescope:
   - The completed Jupyter Notebook (.ipynb). And for video rendering tasks, make sure that the generated videos are inline.

   - An exported PDF version of the notebook. And ensure important content (including the modified code, outputs, and markdown) is visible in the exported PDF.


In [ ]:
# Install necessary libraries
%pip install mujoco==3.5.0 matplotlib

In [ ]:
# Load necessary libs
import mujoco
import mujoco.viewer
import numpy as np
import matplotlib
import time

## 1. Newton's Cradle

In [ ]:
# Helper Functions
from typing import List
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib import animation


def display_images(
    images: List[np.ndarray], dpi=100.0, format="html5_video", fps=None, **kwargs
):
    """Display images as an animation in jupyter notebook.

    Args:
        images: images with equal shape.
        dpi: resolution (dots per inch).
        format (str): one of ["html5_video", "jshtml"]

    References:
        https://gist.github.com/foolishflyfox/e30fd8bfbb6a9cee9b1a1fa6144b209c
        http://louistiao.me/posts/notebooks/embedding-matplotlib-animations-in-jupyter-as-interactive-javascript-widgets/
        https://stackoverflow.com/questions/35532498/animation-in-ipython-notebook/46878531#46878531
    """
    h, w = images[0].shape[:2]
    fig = plt.figure(figsize=(h / dpi, w / dpi), dpi=dpi)
    fig_im = plt.figimage(images[0])
    if fps is not None:
        kwargs["interval"] = 1000 / fps

    def animate(image):
        fig_im.set_array(image)
        return (fig_im,)

    anim = animation.FuncAnimation(fig, animate, frames=images, **kwargs)
    if format == "html5_video":
        # NOTE(jigu): can not show in VSCode
        display(HTML(anim.to_html5_video()))
    elif format == "jshtml":
        display(HTML(anim.to_jshtml()))
    else:
        raise NotImplementedError(format)
    plt.close(fig)


def load_newton_cradle(xml: str):
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    return model, data


def render_newton_cradle(model, data, duration=6, width=480, height=480, fps=24):
    total_frames = int(fps * duration)
    frame_interval = 1.0 / fps
    frames = []
    frames_written = 0
    next_frame_time = 0.0
    renderer = mujoco.Renderer(model, height=height, width=width)
    while frames_written < total_frames:
        # Capture frame whenever simulation time reaches the next frame boundary
        if data.time >= next_frame_time - 1e-9:
            renderer.update_scene(data)
            frame = renderer.render()
            frames.append(frame.copy())
            frames_written += 1
            next_frame_time += frame_interval
        mujoco.mj_step(model, data)
    renderer.close()
    return frames

In [ ]:
newton_cradle_xml = """
<mujoco model="newtons_cradle">
    <compiler angle="degree"/>

    <option timestep="1e-4" gravity="0 0 -9.79"/>

    <default>
        <joint type="hinge" axis="1 0 0" damping="0.0001"/>

        <default class="support">
            <geom type="box" rgba="0.75 0.75 0.75 1" mass="0" contype="0" conaffinity="0"/>
        </default>

        <default class="white_ball">
            <geom type="sphere" size="0.025" mass="0.1" density="8000" rgba="0.95 0.95 0.95 1" condim="1"/>
        </default>

        <default class="string">
            <geom type="cylinder" size="0.001 0.12" rgba="0.5 0.5 0.5 1" mass="0" contype="0" conaffinity="0"/>
        </default>
    </default>

    <asset>
        <texture name="texplane" type="2d" builtin="checker" rgb1=".2 .3 .4" rgb2=".1 .15 .2" width="512" height="512" mark="cross" markrgb=".8 .8 .8"/>
        <material name="matplane" reflectance="0.3" texture="texplane" texrepeat="1 1" texuniform="true"/>
    </asset>

    <worldbody>
        <light directional="true" diffuse=".8 .8 .8" specular=".2 .2 .2" pos="0 0 5" dir="0 0 -1"/>
        <geom name="ground" type="plane" size="2 2 0.1" pos="0 0 0" material="matplane"/>

        <body name="table_base" pos="0 0 0">
            <geom name="table_surface" type="box" size="0.4 0.3 0.02" pos="0 0 0.38" rgba="0.5 0.35 0.2 1"/>
            <geom name="table_leg_front_right" type="box" size="0.02 0.02 0.18" pos="0.36 0.26 0.18" rgba="0.4 0.25 0.1 1"/>
            <geom name="table_leg_front_left" type="box" size="0.02 0.02 0.18" pos="-0.36 0.26 0.18" rgba="0.4 0.25 0.1 1"/>
            <geom name="table_leg_back_right" type="box" size="0.02 0.02 0.18" pos="0.36 -0.26 0.18" rgba="0.4 0.25 0.1 1"/>
            <geom name="table_leg_back_left" type="box" size="0.02 0.02 0.18" pos="-0.36 -0.26 0.18" rgba="0.4 0.25 0.1 1"/>

            <body name="support_stand" pos="0 0 0.40">
                <geom name="base_plate" class="support" size="0.25 0.15 0.01" pos="0 0 0.01"/>

                <geom name="front_left_pillar" class="support" size="0.01 0.01 0.15" pos="-0.22 -0.12 0.17"/>
                <geom name="front_right_pillar" class="support" size="0.01 0.01 0.15" pos="0.22 -0.12 0.17"/>
                <geom name="front_top_beam" class="support" size="0.23 0.01 0.01" pos="0 -0.12 0.33"/>

                <geom name="back_left_pillar" class="support" size="0.01 0.01 0.15" pos="-0.22 0.12 0.17"/>
                <geom name="back_right_pillar" class="support" size="0.01 0.01 0.15" pos="0.22 0.12 0.17"/>
                <geom name="back_top_beam" class="support" size="0.23 0.01 0.01" pos="0 0.12 0.33"/>

                <body name="ball" pos="-0.1 0 0.32">
                    <joint name="j"/>
                    <geom class="string" fromto="0 -0.12 0 0 0 -0.25"/>
                    <geom class="string" fromto="0 0.12 0 0 0 -0.25"/>
                    <geom class="white_ball" pos="0 0 -0.25"/>
                </body>                
            </body>
        </body>
    </worldbody>
</mujoco>
"""

### 1.1 MuJoCo Basics

Read through the MJCF model defined in the cell above and answer the following questions by referring to MuJoCo's [documentation](https://mujoco.readthedocs.io/en/3.5.0/overview.html).

**(a) Body, Geom, and Site** (3 points)

MuJoCo has three fundamental building-block concepts: **body**, **geom**, and **site**. Briefly explain the role and purpose of each:

- What is a **body** and what does it represent in the simulation?
- What is a **geom** and how is it used?
- What is a **site** and how does it differ from a geom?

**(b) Primitive Geometry Sizes** (1 points)

The `size` attribute has different meanings depending on the geometry type. Using the `box` type as an example:

```xml
<geom type="box" size="0.4 0.3 0.02" pos="0 0 0.38" .../>
```

What do the three values in `size="0.4 0.3 0.02"` represent for a box? Explain the meaning of `size` for `sphere`, `cylinder`, and `capsule` as well.

**(c) Coordinate Frames for `pos`** (2 points)

In MuJoCo, every `pos` attribute specifies a position in the parent body's local frame. Treating the `worldbody` frame as the world frame, what is the position of `ball` expressed in the world frame?


In [ ]:
display_images(render_newton_cradle(*load_newton_cradle(newton_cradle_xml), fps=24), fps=24)

### TODO: Solution

### 1.2 Add More Balls (4 points)

The current MJCF model defines only a **single ball**. A real Newton's cradle requires **5 balls** arranged side by side, each suspended by two strings from the support frame. Modify the MJCF model to include 5 balls in total, spaced evenly along the x-axis with an appropriate center-to-center separation. Once done, run the code cell below to verify — you should see 5 balls hanging neatly in a row. And explain your modification here.


In [ ]:
newton_cradle_xml = """
TODO: Your modification here
"""
display_images(render_newton_cradle(*load_newton_cradle(newton_cradle_xml), fps=24), fps=24)

### 1.3 Fix the Joint (2 points)

To set the balls in motion, we apply an initial angle of 30° to the first ball via `data.qpos[0] = np.deg2rad(30)` at the start of the simulation (see the code cell below). However, you will notice that the simulation behaves incorrectly — the ball swings in the wrong direction. Identify the bug and give your solution.


In [ ]:
newton_cradle_xml = """
TODO: Your modification here
"""
model, data = load_newton_cradle(newton_cradle_xml)
data.qpos[0] = np.deg2rad(30)
display_images(render_newton_cradle(model, data, fps=24), fps=24)

### 1.4 Newton's Cradle

After fixing the joint axis in the previous problem, the simulation runs — but the behavior is **physically incorrect**: all 5 balls tend to move together as a clump rather than exhibiting the expected Newton's cradle pattern. Under the assumption of **perfectly elastic collisions**, the correct behavior should resemble [this](https://en.wikipedia.org/wiki/Newton%27s_cradle#/media/File:Newtons_cradle_animation_book_2.gif). The root cause lies in MuJoCo's default [contact/solver parameters](https://mujoco.readthedocs.io/en/3.5.0/modeling.html#solver-parameters).

**(a) Fix the Simulation** (4 points)

Modify the `white_ball` default class (available attributes are listed [here](https://mujoco.readthedocs.io/en/3.5.0/XMLreference.html#body-geom)) to make the contacts as stiff and elastic as possible. Re-run the simulation and verify that, after the ball is released, only the fifth ball swings out on the other side. Explain your solution here.

**(b) Video Rendering** (4 points)

Add a `main_cam` camera to the scene, positioned 1.0 m in front of the cradle (along the −y axis) and 0.75 m high, with a 75° rotation around the x-axis (euler angle). Then implement and run the `render_newton_cradle` function to produce a 3-second video at 24 fps demonstrating the result (the video should be output of the code cell).


In [ ]:
newton_cradle_xml = """
TODO: Your modification here
"""
# TODO: You may need to modify the following function
def render_newton_cradle(model, data, duration=6, width=480, height=480, fps=24):
    total_frames = int(fps * duration)
    frame_interval = 1.0 / fps
    frames = []
    frames_written = 0
    next_frame_time = 0.0
    renderer = mujoco.Renderer(model, height=height, width=width)
    while frames_written < total_frames:
        # Capture frame whenever simulation time reaches the next frame boundary
        if data.time >= next_frame_time - 1e-9:
            renderer.update_scene(data)
            frame = renderer.render()
            frames.append(frame.copy())
            frames_written += 1
            next_frame_time += frame_interval
        mujoco.mj_step(model, data)
    renderer.close()
    return frames
model, data = load_newton_cradle(newton_cradle_xml)
data.qpos[0] = np.deg2rad(30)
display_images(render_newton_cradle(model, data, fps=24), fps=24)

## 2. Domino Chain (20 points)

Please use box `geom`s to build a domino system. The grading will be based on three parts and three difficulty levels:

1. Please build a static domino system that forms the shape of the letter C, P, or B.

2. Provide your initial condition (trigger mechanism) to start the chain reaction (the dominoes may fall, but not necessarily all of them). The restriction here is that only one trigger can be used.

3. Ensure that the chain reaction proceeds smoothly. In particular, when you choose the letter B shape, ensure that **two** chain reactions occur simultaneously. And render (not screen cast) a 8s video with 24fps to demonstrate the result.

Please present the results for each process. Parts 1 and 2 each account for 25% of the score, and part 3 accounts for 50% of the score. If you choose the letters C, P, or B, the maximum achievable scores are 60%, 80%, and 100%, respectively.
